# 部署服务器
​
## 与原始 FastAPI 应用分开部署
您不限于在创建 MCP 的同一 FastAPI 应用上为其提供服务。

你可以从一个 FastAPI 应用创建一个 MCP 服务器，然后将其挂载到另一个应用：



In [ ]:
from fastapi import FastAPI
from fastapi_mcp import FastApiMCP

# Your API app
api_app = FastAPI()
# ... define your API endpoints on api_app ...

# A separate app for the MCP server
mcp_app = FastAPI()

# Create MCP server from the API app
mcp = FastApiMCP(api_app)

# Mount the MCP server to the separate app
mcp.mount(mcp_app)

然后，您可以分别运行这两个应用程序：

```bash
uvicorn main:api_app --host api-host --port 8001
uvicorn main:mcp_app --host mcp-host --port 8000
```

# 刷新服务器
创建 MCP 服务器后添加端点

如果在创建 MCP 服务器后将端点添加到 FastAPI 应用，则需要刷新服务器以包含它们：


In [ ]:
from fastapi import FastAPI
from fastapi_mcp import FastApiMCP

app = FastAPI()

mcp = FastApiMCP(app)
mcp.mount()

# Add new endpoints after MCP server creation
@app.get("/new/endpoint/", operation_id="new_endpoint")
async def new_endpoint():
    return {"message": "Hello, world!"}

# Refresh the MCP server to include the new endpoint
mcp.setup_server()

## Transport
如何与 FastAPI 应用进行通信

FastAPI-MCP 默认使用 ASGI 传输，这意味着它无需发出 HTTP 请求即可直接与您的 FastAPI 应用通信。这种方式更高效，并且不需要基础 URL。

FastAPI 服务器甚至不需要运行。

如果您需要指定自定义基本 URL 或使用不同的传输方法，您可以提供自己的httpx.AsyncClient：

In [ ]:
import httpx
from fastapi import FastAPI
from fastapi_mcp import FastApiMCP

app = FastAPI()

custom_client = httpx.AsyncClient(
    base_url="https://api.example.com",
    timeout=30.0
)

mcp = FastApiMCP(
    app,
    http_client=custom_client
)

mcp.mount()